# Paper Trading: Gap Fade Si (Tinkoff Sandbox)

Paper trading the overnight gap fade strategy on Si futures (MOEX) via Tinkoff Sandbox API. Strategy parameters from walk-forward optimization in `03_walk_forward.ipynb`.

In [10]:
# Imports and strategy parameters
import os
import json
import time
import requests
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, timezone, date
from pathlib import Path

from dotenv import load_dotenv
from tinkoff.invest import (
    Client, CandleInterval, OrderDirection, OrderType,
    MoneyValue, RequestError, InstrumentStatus,
    StopOrderDirection, StopOrderType, StopOrderExpirationType,
)
from tinkoff.invest.sandbox.client import SandboxClient
from tinkoff.invest.utils import now as ti_now

import matplotlib.pyplot as plt
plt.rcParams["figure.figsize"] = (14, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

load_dotenv()
TINKOFF_TOKEN = os.getenv("TINKOFF_TOKEN")
assert TINKOFF_TOKEN, "Token not found, check .env"
print(f"Token loaded: {TINKOFF_TOKEN[:6]}...{TINKOFF_TOKEN[-4:]}")

SI_UID = "b4df2961-035a-4dcd-8372-9af0db10d2c9"  # SiM6
SI_TICKER = "SiM6"
SI_GO = 11_050       # margin requirement
COMMISSION_RT = 3    # round-trip commission
SLIPPAGE_PTS = 10    # slippage
TOTAL_COST = COMMISSION_RT + SLIPPAGE_PTS

GAP_THRESHOLD_PCT = 0.5   # gap threshold (conservative)
STOP_MULTIPLIER = 1.5     # stop = gap_size * 1.5
NUM_CONTRACTS = 1

MOEX_BASE = "https://iss.moex.com/iss"

TRADE_LOG = Path("data/paper_trades.json")
TRADE_LOG.parent.mkdir(parents=True, exist_ok=True)

print(f"\nParams: threshold={GAP_THRESHOLD_PCT}%, stop={STOP_MULTIPLIER}x, contracts={NUM_CONTRACTS}")
print("Ready.")

Token loaded: t.tK8o...4ySQ

Params: threshold=0.5%, stop=1.5x, contracts=1
Ready.


In [12]:
# Sandbox account setup and management
def sandbox_open(initial_rub: int = 100_000) -> str:
    with SandboxClient(TINKOFF_TOKEN) as client:
        resp = client.sandbox.open_sandbox_account()
        account_id = resp.account_id
        client.sandbox.sandbox_pay_in(
            account_id=account_id,
            amount=MoneyValue(currency="rub", units=initial_rub, nano=0),
        )
        print(f"Sandbox account: {account_id}")
        print(f"Balance: {initial_rub:,} RUB (virtual)")
        return account_id


def sandbox_list_accounts():
    with SandboxClient(TINKOFF_TOKEN) as client:
        accounts = client.users.get_accounts()
        for acc in accounts.accounts:
            print(f"  {acc.id} | {acc.name} | opened: {acc.opened_date.date()}")
        return accounts.accounts


def sandbox_close(account_id: str):
    with SandboxClient(TINKOFF_TOKEN) as client:
        client.sandbox.close_sandbox_account(account_id=account_id)
        print(f"Account {account_id} closed")


print("Existing sandbox accounts:")
existing = sandbox_list_accounts()

if not existing:
    print("\nCreating new account...")
    SANDBOX_ID = sandbox_open(100_000)
else:
    SANDBOX_ID = existing[0].id
    print(f"\nUsing existing: {SANDBOX_ID}")

Existing sandbox accounts:
  42f1341f-888a-4629-b848-bee166157c02 |  | opened: 2026-05-27
  87133d90-1330-4316-9a11-0a9f55bb9a5d |  | opened: 2026-06-10
  a1102be4-3a9d-45a7-8eef-27f4ca0d4bfe |  | opened: 2026-06-15
  c527b74d-c442-4d79-81e7-f15f07aa252b |  | opened: 2026-06-15
  d29a1f3c-8b80-46bb-8f2a-4a7f7ac0a31d |  | opened: 2026-06-15

Using existing: 42f1341f-888a-4629-b848-bee166157c02


In [13]:
# Fetch MOEX candles and extract evening close / morning open sessions
def moex_get_candles(security, interval=60, start="2024-01-01", end=None,
                     engine="futures", market="forts"):
    if end is None:
        end = datetime.now().strftime("%Y-%m-%d")
    all_rows, page_start = [], 0
    while True:
        url = f"{MOEX_BASE}/engines/{engine}/markets/{market}/securities/{security}/candles.json"
        params = {"from": start, "till": end, "interval": interval, "start": page_start}
        resp = requests.get(url, params=params, timeout=15)
        resp.raise_for_status()
        candles = resp.json()["candles"]
        rows = candles["data"]
        if not rows:
            break
        all_rows.extend(rows)
        page_start += len(rows)
        time.sleep(0.2)
    if not all_rows:
        return pd.DataFrame()
    df = pd.DataFrame(all_rows, columns=candles["columns"])
    df["begin"] = pd.to_datetime(df["begin"])
    df = df.rename(columns={"begin": "timestamp", "value": "volume_rub", "volume": "vol"})
    return df.sort_values("timestamp").reset_index(drop=True)


def get_last_sessions(ticker=SI_TICKER, days_back=5):
    start = (datetime.now() - timedelta(days=days_back + 3)).strftime("%Y-%m-%d")
    df = moex_get_candles(ticker, interval=60, start=start)
    if df.empty:
        print("No data from MOEX!")
        return pd.DataFrame(), df

    df["date"] = df["timestamp"].dt.date
    df["hour"] = df["timestamp"].dt.hour

    evening = df[df["hour"] >= 19].groupby("date").last()[["close"]].rename(
        columns={"close": "evening_close"})
    morning = df[df["hour"] < 14].groupby("date").first()[["open"]].rename(
        columns={"open": "morning_open"})

    sessions = pd.DataFrame(index=morning.index)
    sessions["morning_open"] = morning["morning_open"]

    ev_dates = list(evening.index)
    ev_values = evening["evening_close"].values
    prev_close = {}
    for i in range(len(ev_dates) - 1):
        next_dates = [d for d in sessions.index if d > ev_dates[i]]
        if next_dates:
            prev_close[next_dates[0]] = ev_values[i]

    sessions["prev_evening_close"] = pd.Series(prev_close)
    sessions = sessions.dropna()
    sessions["gap_points"] = sessions["morning_open"] - sessions["prev_evening_close"]
    sessions["gap_pct"] = sessions["gap_points"] / sessions["prev_evening_close"] * 100

    return sessions, df


sessions, hourly = get_last_sessions(days_back=10)
print(f"Last sessions ({len(sessions)} days):\n")
display(sessions[["morning_open", "prev_evening_close", "gap_points", "gap_pct"]].tail(10))

Last sessions (6 days):



,morning_open,prev_evening_close,gap_points,gap_pct
date,,,,
2026-06-09,73295,73151.0,144.0,0.196853
2026-06-10,72200,72193.0,7.0,0.009696
2026-06-11,72500,72511.0,-11.0,-0.015170
2026-06-15,72834,72527.0,307.0,0.423291
2026-06-16,72250,72239.0,11.0,0.015227
2026-06-17,72780,72817.0,-37.0,-0.050812


In [14]:
# Signal detection: check if today's gap exceeds threshold
def check_signal(sessions_df, target_date=None):
    if target_date is None:
        target_date = date.today()

    if target_date not in sessions_df.index:
        print(f"No data for {target_date}")
        print(f"Last available session: {sessions_df.index[-1]}")
        return None

    row = sessions_df.loc[target_date]
    gap_pct = row["gap_pct"]
    gap_pts = row["gap_points"]
    prev_close = row["prev_evening_close"]
    morning_open = row["morning_open"]

    print(f"{'='*50}")
    print(f"  SIGNAL for {target_date}")
    print(f"{'='*50}")
    print(f"  Evening close (prev day): {prev_close:,.0f}")
    print(f"  Morning open:             {morning_open:,.0f}")
    print(f"  Gap: {gap_pts:+,.0f} pts ({gap_pct:+.2f}%)")
    print()

    if abs(gap_pct) < GAP_THRESHOLD_PCT:
        print(f"  >>> NO SIGNAL (|gap| = {abs(gap_pct):.2f}% < threshold {GAP_THRESHOLD_PCT}%)")
        return None

    gap_size = abs(morning_open - prev_close)

    if gap_pct > 0:
        direction = "SHORT"
        target = prev_close
        stop = morning_open + gap_size * STOP_MULTIPLIER
    else:
        direction = "LONG"
        target = prev_close
        stop = morning_open - gap_size * STOP_MULTIPLIER

    risk = abs(morning_open - stop)
    reward = abs(morning_open - target)

    signal = {
        "date": target_date,
        "direction": direction,
        "entry": morning_open,
        "target": target,
        "stop": stop,
        "gap_pct": gap_pct,
        "gap_pts": gap_pts,
        "risk_pts": risk,
        "reward_pts": reward,
    }

    print(f"  >>> SIGNAL: {direction}")
    print(f"  Entry:  {morning_open:,.0f}")
    print(f"  Target: {target:,.0f} ({reward:,.0f} pts)")
    print(f"  Stop:   {stop:,.0f} ({risk:,.0f} pts)")
    print(f"  R:R =   1:{reward/risk:.1f}")

    return signal


sessions, hourly = get_last_sessions(days_back=10)
signal = check_signal(sessions)

if signal is None:
    print("\n\nRecent signals:")
    for d in sessions.index[-5:]:
        row = sessions.loc[d]
        flag = " <-- SIGNAL" if abs(row["gap_pct"]) >= GAP_THRESHOLD_PCT else ""
        print(f"  {d}: gap {row['gap_pct']:+.2f}% ({row['gap_points']:+,.0f} pts){flag}")

No data for 2026-06-20
Last available session: 2026-06-17


Recent signals:
  2026-06-10: gap +0.01% (+7 pts)
  2026-06-11: gap -0.02% (-11 pts)
  2026-06-15: gap +0.42% (+307 pts)
  2026-06-16: gap +0.02% (+11 pts)
  2026-06-17: gap -0.05% (-37 pts)


In [15]:
# Sandbox order execution and portfolio inspection
def q(money):
    """Quotation/MoneyValue to float."""
    return money.units + money.nano / 1e9


def sandbox_get_portfolio():
    with SandboxClient(TINKOFF_TOKEN) as client:
        p = client.operations.get_portfolio(account_id=SANDBOX_ID)

    total = q(p.total_amount_portfolio)
    rub = q(p.total_amount_currencies)

    print(f"Sandbox portfolio ({SANDBOX_ID[:8]}...):")
    print(f"  Total:      {total:,.0f} RUB")
    print(f"  Cash:       {rub:,.0f} RUB")

    if p.positions:
        print(f"  Positions:")
        for pos in p.positions:
            qty = q(pos.quantity)
            avg = q(pos.average_position_price) if pos.average_position_price else 0
            pnl = q(pos.expected_yield) if pos.expected_yield else 0
            print(f"    {pos.figi}: {qty:+.0f} lots, avg={avg:,.0f}, PnL={pnl:+,.0f}")
    else:
        print(f"  No open positions")

    return p


def sandbox_market_order(direction: str, lots: int = NUM_CONTRACTS):
    dir_map = {
        "buy":  OrderDirection.ORDER_DIRECTION_BUY,
        "sell": OrderDirection.ORDER_DIRECTION_SELL,
    }

    with SandboxClient(TINKOFF_TOKEN) as client:
        order = client.sandbox.post_sandbox_order(
            account_id=SANDBOX_ID,
            instrument_id=SI_UID,
            quantity=lots,
            direction=dir_map[direction],
            order_type=OrderType.ORDER_TYPE_MARKET,
        )

    price = q(order.executed_order_price) if order.executed_order_price else 0
    commission = q(order.executed_commission) if order.executed_commission else 0

    print(f"Order filled:")
    print(f"  {direction.upper()} {lots} x {SI_TICKER}")
    print(f"  Price: {price:,.0f}")
    print(f"  Commission: {commission:,.2f}")
    print(f"  Status: {order.execution_report_status.name}")

    return {
        "order_id": order.order_id,
        "direction": direction,
        "lots": lots,
        "price": price,
        "commission": commission,
        "status": order.execution_report_status.name,
        "time": datetime.now().isoformat(),
    }


def get_current_price():
    with SandboxClient(TINKOFF_TOKEN) as client:
        resp = client.market_data.get_last_prices(instrument_id=[SI_UID])
        if resp.last_prices:
            price = q(resp.last_prices[0].price)
            ts = resp.last_prices[0].time
            print(f"Si current price: {price:,.0f} (at {ts.strftime('%H:%M:%S')})")
            return price
    return None


sandbox_get_portfolio()
print()
get_current_price()

Sandbox portfolio (42f1341f...):
  Total:      100,000 RUB
  Cash:       100,000 RUB
  Positions:
    RUB000UTSTOM: +100000 lots, avg=1, PnL=+0

Si current price: 73,357 (at 15:59:45)


73357.0

In [16]:
# Trading cycle: detect signal, enter, monitor price, exit on target/stop/timeout
def load_trade_log():
    if TRADE_LOG.exists():
        with open(TRADE_LOG) as f:
            return json.load(f)
    return []


def save_trade_log(trades):
    with open(TRADE_LOG, "w") as f:
        json.dump(trades, f, indent=2, default=str)


def execute_gap_fade(dry_run=False):
    sessions, hourly = get_last_sessions(days_back=5)
    signal = check_signal(sessions)

    if signal is None:
        return None

    if dry_run:
        print("\n[DRY RUN -- trade not executed]")
        return signal

    print(f"\n--- Opening position ---")
    if signal["direction"] == "LONG":
        entry_order = sandbox_market_order("buy", NUM_CONTRACTS)
    else:
        entry_order = sandbox_market_order("sell", NUM_CONTRACTS)

    entry_price = entry_order["price"]
    target = signal["target"]
    stop = signal["stop"]
    direction = signal["direction"]

    print(f"\nPosition opened: {direction} @ {entry_price:,.0f}")
    print(f"Target: {target:,.0f} | Stop: {stop:,.0f}")
    print(f"\nMonitoring price (every 60s)...")
    print(f"Ctrl+C to interrupt (position stays open!)\n")

    exit_price = None
    exit_reason = None
    check_count = 0

    try:
        while True:
            price = None
            try:
                with SandboxClient(TINKOFF_TOKEN) as client:
                    resp = client.market_data.get_last_prices(instrument_id=[SI_UID])
                    if resp.last_prices:
                        price = q(resp.last_prices[0].price)
            except Exception as e:
                print(f"  Price fetch error: {e}")
                time.sleep(60)
                continue

            now = datetime.now()
            check_count += 1

            if direction == "LONG":
                unrealized = price - entry_price
            else:
                unrealized = entry_price - price

            print(f"  [{now.strftime('%H:%M:%S')}] Price: {price:,.0f} | "
                  f"PnL: {unrealized:+,.0f} pts | "
                  f"Target: {target:,.0f} | Stop: {stop:,.0f}")

            if direction == "LONG" and price >= target:
                exit_price = price
                exit_reason = "target"
                break
            elif direction == "SHORT" and price <= target:
                exit_price = price
                exit_reason = "target"
                break

            if direction == "LONG" and price <= stop:
                exit_price = price
                exit_reason = "stop"
                break
            elif direction == "SHORT" and price >= stop:
                exit_price = price
                exit_reason = "stop"
                break

            if now.hour >= 19:
                exit_price = price
                exit_reason = "timeout"
                break

            time.sleep(60)

    except KeyboardInterrupt:
        print("\n\nInterrupted by user!")
        print("Position still open. Close manually:")
        close_dir = "sell" if direction == "LONG" else "buy"
        print(f'  sandbox_market_order("{close_dir}", {NUM_CONTRACTS})')
        return signal

    print(f"\n--- Closing position ({exit_reason}) ---")
    if direction == "LONG":
        close_order = sandbox_market_order("sell", NUM_CONTRACTS)
    else:
        close_order = sandbox_market_order("buy", NUM_CONTRACTS)

    actual_exit = close_order["price"]

    if direction == "LONG":
        pnl_gross = actual_exit - entry_price
    else:
        pnl_gross = entry_price - actual_exit

    pnl_net = pnl_gross - TOTAL_COST

    trade_record = {
        "date": str(signal["date"]),
        "direction": direction,
        "entry_price": entry_price,
        "exit_price": actual_exit,
        "target": target,
        "stop": stop,
        "gap_pct": signal["gap_pct"],
        "exit_reason": exit_reason,
        "pnl_gross": pnl_gross,
        "pnl_net": pnl_net,
        "entry_order_id": entry_order["order_id"],
        "exit_order_id": close_order["order_id"],
        "entry_commission": entry_order["commission"],
        "exit_commission": close_order["commission"],
    }

    log = load_trade_log()
    log.append(trade_record)
    save_trade_log(log)

    print(f"\n{'='*50}")
    print(f"  RESULT")
    print(f"{'='*50}")
    print(f"  Direction: {direction}")
    print(f"  Entry:  {entry_price:,.0f}")
    print(f"  Exit:   {actual_exit:,.0f} ({exit_reason})")
    print(f"  PnL gross: {pnl_gross:+,.0f} pts")
    print(f"  PnL net:   {pnl_net:+,.0f} pts (after costs)")
    print(f"  Logged to {TRADE_LOG}")

    return trade_record


# Dry run first to verify signal:
result = execute_gap_fade(dry_run=True)

# To execute live in sandbox, uncomment:
# result = execute_gap_fade(dry_run=False)

No data for 2026-06-20
Last available session: 2026-06-17


In [18]:
# Dashboard: paper trading results vs backtest expectations
def show_dashboard():
    log = load_trade_log()

    if not log:
        print("Trade log empty. No trades yet.")
        print("Run execute_gap_fade(dry_run=False) for the first trade.")
        return

    df = pd.DataFrame(log)
    df["date"] = pd.to_datetime(df["date"])
    n = len(df)

    wins = (df["pnl_net"] > 0).sum()
    total_pnl = df["pnl_net"].sum()
    avg_pnl = df["pnl_net"].mean()

    cum = df["pnl_net"].cumsum()
    max_dd = (cum.cummax() - cum).max()

    BT_WIN_RATE = 87.8
    BT_AVG_PNL = 3131

    print(f"{'='*60}")
    print(f"  PAPER TRADING DASHBOARD")
    print(f"{'='*60}")
    print(f"  Trades:         {n}")
    print(f"  Win rate:       {wins/n*100:.1f}%  (backtest: {BT_WIN_RATE}%)")
    print(f"  PnL total:      {total_pnl:+,.0f} RUB")
    print(f"  PnL avg:        {avg_pnl:+,.0f} RUB/trade  (backtest: {BT_AVG_PNL:+,})")
    print(f"  Max drawdown:   {max_dd:,.0f} RUB")
    print(f"  Exits: target {(df['exit_reason']=='target').mean()*100:.0f}% | "
          f"stop {(df['exit_reason']=='stop').mean()*100:.0f}% | "
          f"timeout {(df['exit_reason']=='timeout').mean()*100:.0f}%")

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

    equity = df["pnl_net"].cumsum()
    ax1.plot(df["date"], equity, "o-", linewidth=1.5, color="#2ecc71", markersize=5)
    ax1.fill_between(df["date"], equity, alpha=0.15, color="#2ecc71")
    ax1.axhline(0, color="black", linewidth=0.5)
    ax1.set_title(f"Paper Trading Equity (n={n}, WR={wins/n*100:.0f}%)")
    ax1.set_ylabel("Cumulative PnL, RUB")

    colors = ["#2ecc71" if p > 0 else "#e74c3c" for p in df["pnl_net"]]
    ax2.bar(range(n), df["pnl_net"], color=colors, alpha=0.8)
    ax2.axhline(0, color="black", linewidth=0.5)
    ax2.axhline(BT_AVG_PNL, color="blue", linewidth=1, linestyle="--", alpha=0.5, label=f"BT avg ({BT_AVG_PNL:+,})")
    ax2.set_title("PnL per Trade")
    ax2.set_xlabel("Trade #")
    ax2.set_ylabel("PnL, RUB")
    ax2.legend()

    plt.tight_layout()
    plt.show()

    print(f"\nTrade details:")
    display(df[["date", "direction", "entry_price", "exit_price", "exit_reason", "gap_pct", "pnl_gross", "pnl_net"]])


show_dashboard()

Trade log empty. No trades yet.
Run execute_gap_fade(dry_run=False) for the first trade.


## Results

**Go-live criteria** (all must be met over 20+ trades):
- Win rate >= 70% (backtest: 87.8%)
- Avg PnL/trade > 1,000 RUB
- Max drawdown < 30,000 RUB

At ~12 signals/month, expect ~2 months of paper trading before live deployment with 1 contract.